## We try here to intercalibrate the detectors
Each TES might have a different gain and we want to align them to the same response.
This calibration might have to be done before stacking the maps of the same TES and different observations because the Moon doesn't have the same emissivity for all Moon phase angles.

In [ ]:
# Allows the modification of the files imported without having to restart kernel or re-import them
%load_ext autoreload
%autoreload 2
# to have interactive plots : pip install ipympl
%matplotlib ipympl

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
import emcee
import pickle
import os

from qubic.lib import Qdictionary, Qinstrument
from qubic.lib.Instrument.Qinstrument import QubicMultibandInstrument
from qubic.lib.Qscene import QubicScene
from qubicpack.utilities import Qubic_DataDir

import pipeline_moon_functions as pmf
import pipeline_moon_plotting as pmp

## We read the maps (here stacked maps because otherwise full of holes)

In [ ]:
datadir_MoonProject = "/Users/huchet/Documents/code/scripts/MoonProject/"

In [ ]:
ang_size = 40 # degrees
xsize = 201
reso = ang_size*60/xsize # arcmin/pix
min_plot = -5e3
max_plot = 1.2e4
zenith = np.array([0, 90])
moon_angsize = np.radians(0.5)

In [ ]:
# Parameters of the maps depending on the observation
list_of_observations = [                  "2026-07-23_rise",
                        "2026-07-24_set", "2026-07-24_rise",
                        "2026-07-25_set", "2026-07-25_rise",
                        "2026-07-26_set", "2026-07-26_rise",
                        "2026-07-27_set", "2026-07-27_rise",
                        "2026-07-28_set"]

dict_already_computed_tshifts = {                         "2026-07-23_rise": -0.63,
                                 "2026-07-24_set": -0.65, "2026-07-24_rise": -0.63,
                                 "2026-07-25_set": -0.7,  "2026-07-25_rise": -0.6,
                                 "2026-07-26_set": -0.52, "2026-07-26_rise": -0.7,
                                 "2026-07-27_set": -0.65, "2026-07-27_rise": -0.85,
                                 "2026-07-28_set": -0.75}
n_obs = len(list_of_observations)
det_los = True # if we want to stack the maps with each detector line of sight at zenith
boresight_angle = -3.2

In [ ]:
# it is too heavy to load all full maps so we choose a submap centered on zenith with a given radius
center = zenith
radius = 30 # degrees
nside = 512
vec_centre = hp.ang2vec(center[0], center[1], lonlat=True)
chosen_patch = hp.query_disc(nside, vec_centre, np.radians(radius))
n_pix = len(chosen_patch)
n_TES = 256
# in order to go from patch to full map
remap = pmf.get_remapping_operator(qubic_patch=chosen_patch, shape_patch=np.shape(chosen_patch), shape_full=hp.nside2npix(nside))
    

# the TES for display, but the computation is done for all
TESNum = 218 #96
i_TES = TESNum - 1
allmaps_ = np.zeros((n_obs, n_TES, n_pix))

det_zen = ""
if det_los:
    det_zen += "_det"

for i_obs, obsname in enumerate(list_of_observations):
    ObsDate, scanning = obsname.split("_")
    tshift = dict_already_computed_tshifts[obsname]
    # file_name = "allmaps-{}_{}_nside{}_zenith{}_tshift{}.pkl".format(ObsDate, scanning, nside, det_zen, tshift) # it used to be boresight_angle = -3.5° for all
    file_name = "allmaps-{}_{}_nside{}_zenith{}_tshift{}_boresight{}.pkl".format(ObsDate, scanning, nside, det_zen, tshift, boresight_angle)
    allTESNum, allmaps = pickle.load( open( datadir_MoonProject + file_name, "rb" ) )
    allmaps_[i_obs] = allmaps[:, chosen_patch]
    print("Read {}".format(file_name))

## We measure the brightness temperature or the emissivity of the Moon

In [ ]:
# in case we use a "flat" Moon
def top_hat_kernel(b, radius): # we create a new kernel for the convolution
    return np.where(np.abs(b) <= radius, 1, 0)
def gaussian_pdf(x, mu, sigma):
    return 1 / (sigma * np.sqrt(2 * np.pi)) * np.exp(-(x - mu)**2 / (2 * sigma**2))

In [ ]:
# angle = np.linspace(-np.pi/64, np.pi/64, 1000)
# moon_flat = top_hat_kernel(angle, moon_angsize/2) # we need radius not diameter
# beam = gaussian_pdf(angle, 0, np.radians(0.68)/pmf.conv_reso_fwhm)
# npoints = int(10*60/reso) # 10 degrees seen with a resolution of reso arcmin
# npoints += (1 - npoints%2)
npoints = 10000
print(npoints)
angle = np.linspace(-5, 5, npoints) # degrees
moon_flat = top_hat_kernel(angle, np.degrees(moon_angsize)/2) # we need radius not diameter
beam = gaussian_pdf(angle, 0, 0.68/pmf.conv_reso_fwhm)
convolved = np.convolve(moon_flat, beam, mode="same")
# normalisation for plot
# moon_flat = moon_flat/(np.sum(moon_flat)*(angle[1] - angle[0]))
# convolved = convolved/(np.sum(convolved)*(angle[1] - angle[0]))

In [ ]:
print(np.max(beam)/2)
print(0.68/pmf.conv_reso_fwhm)
print(np.sum(beam)*(angle[1] - angle[0]))

In [ ]:
fwhm_beam = 2 * np.mean(np.abs(np.degrees(angle)[np.isclose(beam, np.max(beam)/2, atol=1e-8, rtol = 1e-2)]))
fwhm_convolved = 2 * np.mean(np.abs(np.degrees(angle)[np.isclose(convolved, np.max(convolved)/2, atol=1e-8, rtol = 1e-2)]))

print("fwhm_beam", fwhm_beam)
print("fwhm_convolved", fwhm_convolved)

fwhm_test = np.sqrt(fwhm_beam**2 + np.degrees(moon_angsize)**2)
print("fwhm_test", fwhm_test)

In [ ]:
sigma_beam = fwhm_beam/pmf.conv_reso_fwhm
sigma_convolved = fwhm_convolved/pmf.conv_reso_fwhm
print("sigma_beam", sigma_beam)
print("sigma_convolved", sigma_convolved)

In [ ]:
plt.figure()
plt.plot(np.degrees(angle), moon_flat, label="moon")
plt.plot(np.degrees(angle), beam, label="beam")
plt.plot(np.degrees(angle), convolved, label="convolution")
for fact in [-1, 1]:
    plt.axvline(x=fact*fwhm_beam/2, ls="dotted", c="orange")
    plt.axvline(x=fact*sigma_beam, ls="--", c="orange")
    plt.axvline(x=fact*fwhm_convolved/2, ls="dotted", c="green")
    plt.axvline(x=fact*sigma_convolved, ls="--", c="green")
plt.legend()
plt.show()

In [ ]:
# we will take the disc-averaged signal
radius_moon = 2 * np.radians(sigma_convolved) # radius of the disc at 95% of total power
print(np.degrees(radius_moon))
vec_centre = hp.ang2vec(theta=0, phi=90, lonlat=True)
moon_disk = hp.query_disc(nside=nside, vec=vec_centre, radius=radius_moon)

In [ ]:
i_obs = 0
i_TES = 95

hp.gnomview(remap(allmaps_[i_obs, i_TES]), rot=zenith, reso=reso, min=min_plot, max=max_plot)
phis = np.radians(np.linspace(0, 360, 200))
thetas = np.full_like(phis, radius_moon)
hp.projplot(thetas, phis, lonlat=False, color="r")
plt.show()

In [ ]:
full_map = remap(allmaps_[i_obs, i_TES])
moon_pix = full_map[moon_disk]
print(np.shape(moon_pix))
print(np.min(moon_pix), np.max(moon_pix))
bad_pix = moon_pix == hp.UNSEEN
avg_moon = np.mean(moon_pix[~bad_pix])
print(avg_moon)
med_mon = np.median(moon_pix[~bad_pix])
print(med_mon)

full_map[moon_disk] = 0

In [ ]:
hp.gnomview(full_map, rot=zenith, reso=reso, min=min_plot, max=max_plot)
plt.show()

In [ ]:
print(np.shape(allmaps))

In [ ]:
# %%script echo skipping

# We perform the Moon position fit and show the result for one TES

### Pick a TES number
mynum = 96 #64 #170 #71 #211 #179 #61 #96 #247 # 152
# print("TES {} is {} in visibly_ok_arr".format(mynum, visibly_ok_arr[mynum - 1]))

myrotinit = np.array([0, 90, 0]) # zenith is better
zenith = np.array([0, 90, 0])

idx = np.where(np.array(allTESNum)==mynum)[0][0]
xs = 301
reso = 6
pack = None

# if working on maps at zenith for each detector
resfit, mapxy, mapfit, extent, fig, ijres, ijerr = pmf.fit_one_tes(allmaps[idx, :], xs, reso, rot=myrotinit, return_images=True, renorm=True, doplot=True, azelguess=zenith, verbose=True, distok=25)

plt.show()
# sys.exit()

In [ ]:
# %%script echo skipping

# We then perform it for all TES and store the results

# print(allTESNum)
# print(np.shape(allmaps))

reso = 6 #4 #5
xs = 301
distok = 25 # pixels!
allamp = np.zeros(len(allTESNum))
allerramp = np.zeros(len(allTESNum))
allFWHM = np.zeros(len(allTESNum)) * np.nan
allerrFWHM = np.zeros(len(allTESNum))
allpos_zen = np.zeros((len(allTESNum), 2)) * np.nan
allerr_zen = np.zeros((len(allTESNum), 2))

moon_fit = []

i_obs = 0

for i in range(len(allTESNum)):
    print(allTESNum[i])
    if det_los: # the maps are centered on the det los
        resfit, ijres, ijerr = pmf.fit_one_tes(remap(allmaps_[i_obs, i, :]), xs, reso, rot=zenith, azelguess=zenith, verbose=False, renorm=True, doplot=False, distok=distok)
    else:
        raise ValueError("We need the maps to be centered on detector line of sight.")
    # if we work on maps at zenith for each detector
    # resfit, ijres, ijerr = pmf.fit_one_tes(allmaps[i,:], xs, reso, rot=zenith, azelguess=zenith, verbose=False, renorm=True, doplot=False, distok=distok)
    allFWHM[i] = resfit.values[3] * pmf.conv_reso_fwhm
    allerrFWHM[i] = resfit.errors[3] * pmf.conv_reso_fwhm
    allamp[i] = resfit.values[0] * pmf.conv_reso_fwhm
    allerramp[i] = resfit.errors[0] * pmf.conv_reso_fwhm
    allpos_zen[i, :] = [ijres[1], ijres[0]] # azt, eltc
    allerr_zen[i, :] = [ijerr[1], ijerr[0]]
    print('TES#{0}: FWHM = {1:5.2f}'.format(i + 1, resfit.values[3] * pmf.conv_reso_fwhm))
    moon_fit.append([allpos_zen[i, :], allFWHM[i], allamp[i]])

In [ ]:
# dictonary initialisation
global_dir = Qubic_DataDir(datafile='instrument.py', datadir='../')
dictfilename = global_dir + '/dicts/pipeline_MoonSalta.dict'
d = Qdictionary.qubicDict()
d.read_from_file(dictfilename)
d['nside'] = 512 #256
d['config'] = 'TD' # no calfile for TD, or even for FI at 150 GHz
d['kind'] = 'I'
d['nf_sub'] = 120 # monoband not implemented on this branch
d['nf_recon'] = 2
d['MultiBand'] = True
d['sweeping_pointing_deadtime'] = False
d['noiseless'] = True
d["use_synthbeam_fits_file"] = False
d["n_nu_fit"] = d['nf_sub'] # monoband not implemented on this branch, but seems like all nsub are in the right band
d['detector_tau'] = 0.01

# scene and instrument initialisations
q_scene = QubicScene(d)
q_instrument = QubicMultibandInstrument(d)

n_nus = len(q_instrument)
ndets = 248
val_0_allnus = np.zeros((ndets, n_nus))
nus = np.zeros(n_nus)
filter_bws = np.zeros(n_nus)
for i_nu in range(n_nus):
    thetas, phis, vals = Qinstrument.QubicInstrument._peak_angles_unsorted(q_scene, q_instrument[i_nu].filter.nu,
                q_instrument[i_nu].detector.center, q_instrument[i_nu].synthbeam, q_instrument[i_nu].horn, q_instrument[i_nu].primary_beam)
    val_0_allnus[:, i_nu] = vals[:, 0]
    nus[i_nu] = q_instrument[i_nu].filter.nu
    filter_bws[i_nu] = q_instrument[i_nu].filter.relative_bandwidth

In [ ]:
TESNum = 96 #163 #96
moon_fit_renorm = np.zeros(256)
for indexQP in range(256):
    indexQS = pmf.iQP2iQS(indexQP)
    print(indexQS)
    if indexQS >= 1000:
        print("thermometer")
        continue
    theory = np.sum(val_0_allnus[indexQS] * filter_bws)
    moon_fit_renorm[indexQP] = moon_fit[indexQP][2] / theory

In [ ]:
print(moon_fit_renorm)

In [ ]:
plt.figure()
plt.plot(allTESNum, moon_fit_renorm)
plt.show()